# 01 — Research Problem and Synthetic Data Design

## Purpose

This notebook establishes the **prediction problem** and constructs a synthetic transactional loan dataset that reproduces the behavioral structure needed for the later research stages.

The emphasis here is not on model training. It is on making the data-generating assumptions explicit and reproducible.

### Research question

> **Can repayment behavior observed during approximately the first third of a loan's lifecycle provide useful information about whether the loan will eventually default?**

### Prediction constraint

The eventual outcome is known only after the loan lifecycle:

- `0` = eventually settles within the original duration
- `1` = does not settle within the original duration

The model, however, must make its prediction using **early-life information only**.

This distinction will be enforced throughout the later notebooks.

## 1. Design principles

The synthetic data is designed to reproduce the *analytical characteristics* of a real transactional lending problem without reproducing any confidential production schema.

The generator includes:

- heterogeneous repayment frequencies;
- varying loan amounts and durations;
- borrower-level prior exposure;
- early missed-payment behavior;
- consecutive missed payments;
- repayment recovery after missed cycles;
- overdue persistence;
- noisy transaction observations;
- a binary eventual-default outcome.

The important design choice is that the **behavioral process generates the outcome**. The target is therefore not assigned independently of the features.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

N_BORROWERS = 12_000
N_LOANS = 30_000

OUTPUT_DIR = Path("../data/synthetic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

## 2. Generate borrower-level heterogeneity

Borrowers are given a small amount of persistent history so that repeated borrowing and prior exposure can later be used as candidate behavioral features.

The values below are **synthetic modeling assumptions**, not estimates of any real institution.

In [ ]:
borrowers = pd.DataFrame({
    "borrower_id": np.arange(1, N_BORROWERS + 1),
    "risk_propensity": rng.normal(0, 1, N_BORROWERS),
    "prior_loan_count": rng.poisson(2.2, N_BORROWERS),
})

borrowers["prior_loan_count"] = borrowers["prior_loan_count"].clip(0, 12)

borrowers.head()

## 3. Generate loan characteristics

Repayment frequency is intentionally heterogeneous.

The synthetic process uses:

- weekly loans,
- bi-weekly loans,
- monthly loans.

This matters because a duration measured in calendar days has different meaning under different installment schedules.

In [ ]:
frequency_names = np.array(["Weekly", "Bi-weekly", "Monthly"])
frequency_days = {"Weekly": 7, "Bi-weekly": 14, "Monthly": 28}

loan_borrower_idx = rng.integers(0, N_BORROWERS, N_LOANS)
frequency = rng.choice(
    frequency_names,
    size=N_LOANS,
    p=[0.50, 0.30, 0.20]
)

loan_duration_days = np.where(
    frequency == "Weekly",
    rng.choice([84, 98, 112, 140], N_LOANS),
    np.where(
        frequency == "Bi-weekly",
        rng.choice([84, 112, 140, 168], N_LOANS),
        rng.choice([112, 140, 168, 196, 224], N_LOANS)
    )
)

disbursed_amount = np.round(
    np.exp(rng.normal(np.log(30_000), 0.55, N_LOANS)),
    0
)

interest_rate = np.round(
    np.clip(rng.normal(0.22, 0.05, N_LOANS), 0.08, 0.40),
    4
)

loans = pd.DataFrame({
    "loan_id": np.arange(1, N_LOANS + 1),
    "borrower_id": borrowers.loc[loan_borrower_idx, "borrower_id"].to_numpy(),
    "frequency_name": frequency,
    "original_loan_duration_days": loan_duration_days,
    "disbursed_amount": disbursed_amount,
    "interest_rate": interest_rate,
})

loans = loans.merge(
    borrowers[["borrower_id", "prior_loan_count", "risk_propensity"]],
    on="borrower_id",
    how="left"
)

loans.head()

## 4. Create repayment schedules

Each loan is converted into scheduled installment dates.

A loan with a monthly schedule therefore has a different number of scheduled observations than a weekly loan with a similar calendar duration.

This creates the structural reason for later investigating **cycle-normalized behavioral features**.

In [ ]:
loans["installment_days"] = loans["frequency_name"].map(frequency_days)
loans["installment_scope_no"] = (
    loans["original_loan_duration_days"] // loans["installment_days"]
).astype(int)

loans["installment_amount"] = np.round(
    loans["disbursed_amount"] * (1 + loans["interest_rate"]) /
    loans["installment_scope_no"],
    0
)

loans[[
    "loan_id",
    "frequency_name",
    "original_loan_duration_days",
    "installment_scope_no",
    "disbursed_amount",
    "installment_amount"
]].head()

## 5. Generate latent repayment behavior

The synthetic process now creates a behavioral risk tendency.

Default probability is influenced by factors such as:

- borrower risk propensity;
- previous borrowing exposure;
- early missed-payment tendency.

This is intentionally simplified. The purpose is not to claim that these are the only drivers of default, but to create a controlled environment in which later modeling methods can be evaluated.

In [ ]:
# Latent risk score used only to generate the synthetic outcome.
# Later notebooks must not treat this hidden variable as an available model feature.

early_risk_score = (
    0.95 * loans["risk_propensity"].to_numpy()
    - 0.08 * np.minimum(loans["prior_loan_count"].to_numpy(), 8)
    + rng.normal(0, 0.45, N_LOANS)
)

base_default_probability = 1 / (1 + np.exp(-(early_risk_score - 1.0)))

# Keep the overall synthetic problem meaningfully imbalanced.
base_default_probability *= 0.75

loans["latent_risk_score"] = early_risk_score
loans["base_default_probability"] = base_default_probability

loans[["latent_risk_score", "base_default_probability"]].describe()

## 6. Generate transactional repayment behavior

The generator produces one observation per scheduled installment.

At each installment:

- some borrowers pay on time;
- some miss a scheduled payment;
- some recover the missed payment later;
- some accumulate consecutive missed installments.

The probability of a missed installment increases with latent risk.

The **final default label is generated from the accumulated repayment trajectory**, rather than directly from an independent random draw.

In [ ]:
rows = []

for row in loans.itertuples(index=False):
    n_installments = int(row.installment_scope_no)

    # Early-life behavior is intentionally influenced by latent risk.
    risk_component = 1 / (1 + np.exp(-(row.latent_risk_score - 0.2)))

    missed_flags = []
    recovery_delays = []
    overdue_days = []

    consecutive = 0

    for installment_no in range(1, n_installments + 1):
        early_weight = 1.25 if installment_no <= max(1, int(np.ceil(n_installments / 3))) else 1.0

        miss_probability = np.clip(
            0.05 + 0.32 * risk_component * early_weight,
            0.02,
            0.75
        )

        missed = rng.random() < miss_probability

        if missed:
            consecutive += 1

            # Some missed payments are recovered quickly; others persist.
            recovery = rng.choice(
                [1, 2, 3, 5],
                p=[0.42, 0.30, 0.18, 0.10]
            )

            overdue = row.installment_days * recovery
        else:
            consecutive = 0
            recovery = 0
            overdue = 0

        missed_flags.append(int(missed))
        recovery_delays.append(int(recovery))
        overdue_days.append(int(overdue))

        rows.append({
            "loan_id": row.loan_id,
            "borrower_id": row.borrower_id,
            "installment_no": installment_no,
            "frequency_name": row.frequency_name,
            "installment_days": row.installment_days,
            "installment_amount": row.installment_amount,
            "missed_installment": int(missed),
            "recovery_delay_cycles": int(recovery),
            "overdue_days": int(overdue),
            "consecutive_missed_so_far": int(consecutive),
        })

transactions = pd.DataFrame(rows)

transactions.shape, transactions.head()

## 7. Derive the eventual default outcome

The target represents the **eventual loan outcome**, not the early behavior itself.

A loan is classified as default when its simulated repayment trajectory contains sufficiently persistent delinquency.

This creates the intended asymmetry:

- the target is based on the full lifecycle;
- predictive features will later be restricted to the first third of that lifecycle.

That separation is essential for avoiding information leakage.

In [ ]:
loan_behavior = (
    transactions
    .groupby("loan_id", as_index=False)
    .agg(
        total_missed_installments=("missed_installment", "sum"),
        max_consecutive_missed=("consecutive_missed_so_far", "max"),
        max_overdue_days=("overdue_days", "max"),
        total_overdue_days=("overdue_days", "sum"),
        total_recovery_delay_cycles=("recovery_delay_cycles", "sum"),
    )
)

loans = loans.merge(loan_behavior, on="loan_id", how="left")

# Default is generated from persistent repayment stress.
default_logit = (
    -2.0
    + 0.20 * loans["total_missed_installments"]
    + 0.16 * loans["max_consecutive_missed"]
    + 0.008 * loans["max_overdue_days"]
    + 0.15 * loans["risk_propensity"]
)

default_probability = 1 / (1 + np.exp(-default_logit))
loans["is_good_or_bad"] = (rng.random(N_LOANS) < default_probability).astype(int)

loans["is_good_or_bad"].value_counts(normalize=True).sort_index()

## 8. Create the early-observation boundary

The prediction point is approximately one-third of the original loan lifecycle.

This notebook creates the boundary explicitly so later notebooks can enforce it.

The most important rule is:

> **Full-lifecycle information may be used to define the target, but only early-lifecycle information may be used as model predictors.**

The variables `total_missed_installments`, `max_consecutive_missed`, `max_overdue_days`, and `total_overdue_days` above are therefore **not yet valid prediction features**; they summarize the complete lifecycle and are shown here only to construct the synthetic outcome.

In [ ]:
loans["prediction_installment"] = np.ceil(
    loans["installment_scope_no"] / 3
).astype(int)

early_transactions = transactions.merge(
    loans[["loan_id", "prediction_installment"]],
    on="loan_id",
    how="left"
)

early_transactions = early_transactions[
    early_transactions["installment_no"] <= early_transactions["prediction_installment"]
].copy()

early_features = (
    early_transactions
    .groupby("loan_id", as_index=False)
    .agg(
        early_missed_installment_count=("missed_installment", "sum"),
        early_max_consecutive_missed=("consecutive_missed_so_far", "max"),
        early_max_overdue_days=("overdue_days", "max"),
        early_total_overdue_days=("overdue_days", "sum"),
        early_recovery_delay_cycles=("recovery_delay_cycles", "sum"),
    )
)

modeling_base = loans[[
    "loan_id",
    "borrower_id",
    "frequency_name",
    "original_loan_duration_days",
    "installment_scope_no",
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "is_good_or_bad",
    "prediction_installment",
]].merge(
    early_features,
    on="loan_id",
    how="left"
)

modeling_base.head()

## 9. Candidate behavioral representations

The synthetic data now supports the same type of behavioral questions that motivate the broader project.

For example:

### Missed-installment behavior
- How many installments were missed during the early observation period?
- Were the misses consecutive?

### Overdue persistence
- How many days did overdue status persist?
- Should calendar-day overdue duration be normalized by installment frequency?

### Relative delinquency
- What proportion of scheduled repayment was overdue?
- Does a relative measure behave more consistently across repayment schedules?

These questions will be investigated formally in later notebooks.

In [ ]:
modeling_base["pass_due_cycle_ratio"] = (
    modeling_base["early_max_overdue_days"]
    / modeling_base["frequency_name"].map(frequency_days)
)

modeling_base["overdue_installment_equivalent"] = (
    modeling_base["early_total_overdue_days"]
    / modeling_base["frequency_name"].map(frequency_days)
)

modeling_base["missed_installment_proportion"] = (
    modeling_base["early_missed_installment_count"]
    / modeling_base["prediction_installment"]
)

modeling_base["overdue_amount_proxy"] = (
    modeling_base["early_missed_installment_count"]
    * modeling_base["installment_amount"]
)

modeling_base["overdue_proportion"] = (
    modeling_base["overdue_amount_proxy"]
    / (
        modeling_base["prediction_installment"]
        * modeling_base["installment_amount"]
    )
).clip(0, 1)

modeling_base[[
    "pass_due_cycle_ratio",
    "overdue_installment_equivalent",
    "missed_installment_proportion",
    "overdue_proportion",
    "is_good_or_bad"
]].head()

## 10. Basic validation

Before moving to EDA, verify that:

1. every modeling record has an eventual outcome;
2. prediction-time features only use early transactions;
3. repayment frequencies are represented;
4. the target is meaningfully imbalanced;
5. the synthetic data contains variation rather than deterministic rules.

These checks are deliberately simple. More rigorous data-quality investigations will be handled in later notebooks.

In [ ]:
checks = {
    "loans": len(loans),
    "transactions": len(transactions),
    "modeling_rows": len(modeling_base),
    "missing_target": int(modeling_base["is_good_or_bad"].isna().sum()),
    "frequencies": modeling_base["frequency_name"].nunique(),
    "default_rate": modeling_base["is_good_or_bad"].mean(),
}

pd.Series(checks)

## 11. Save the reproducible synthetic data

The generated files are intentionally simple CSVs so that subsequent notebooks can be run without proprietary infrastructure.

The next stage will use these files for **EDA and hypothesis generation**.

> **Important:** The synthetic numbers produced by this notebook are illustrative. They should not be interpreted as estimates of any real microfinance portfolio.

In [ ]:
loans.to_csv(OUTPUT_DIR / "synthetic_loans_full.csv", index=False)
transactions.to_csv(OUTPUT_DIR / "synthetic_transactions.csv", index=False)
modeling_base.to_csv(OUTPUT_DIR / "synthetic_early_modeling_base.csv", index=False)

print("Saved:")
for p in [
    OUTPUT_DIR / "synthetic_loans_full.csv",
    OUTPUT_DIR / "synthetic_transactions.csv",
    OUTPUT_DIR / "synthetic_early_modeling_base.csv",
]:
    print(" -", p)

## 12. What this notebook establishes

This first notebook establishes the research setting rather than claiming a modeling result.

### Established

- a binary eventual-default target;
- an explicit one-third-lifecycle prediction boundary;
- heterogeneous repayment schedules;
- a transactional repayment process;
- behavioral signals that can be observed early;
- a reproducible synthetic data-generating process.

### Next research stage

The next notebook will investigate the generated data through **exploratory analysis and hypothesis generation**, including questions around:

- missed-installment behavior;
- consecutive misses;
- overdue persistence;
- repayment recovery;
- normalized overdue measures;
- relative overdue amount.

The purpose is to determine which behavioral patterns deserve formal statistical investigation and eventual inclusion as model features.